# HMS EEG — EDA Overview (WORKSHOP VERSION, self-contained)

This notebook covers the data-understanding groundwork for the rest of the workshop:

1. Dataset scale (patients, recordings, labeled windows)
2. Expert consensus quality — many labels are ambiguous, not clean ground truth
3. Class imbalance — in the full dataset, and how the workshop subset differs
4. Why we split by `patient_id`, not by row
5. Why the labeled window's exact position matters (`eeg_label_offset_seconds`)
6. Raw signal amplitude — the actual numbers behind the mu-law calibration story
7. Electrode layout — which electrodes bipolar montage uses, and which it discards
8. A visual look at one EEG + spectrogram example per class

Fully self-contained — no external .py imports.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

IS_KAGGLE = os.path.exists('/kaggle')
print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'}")

In [ ]:
# ============ Config ============
if IS_KAGGLE:
    DATA_ROOT       = '/kaggle/input/competitions/hms-harmful-brain-activity-classification'
    RAW_TRAIN_PATH  = os.path.join(DATA_ROOT, 'train.csv')
    EEG_DIR         = os.path.join(DATA_ROOT, 'train_eegs')
    SPEC_DIR        = os.path.join(DATA_ROOT, 'train_spectrograms')
    SAMPLE_IDS_PATH = '/kaggle/input/datasets/xiaosufrankhu/midas-summer-academy-wk3-eeg/workshop_sample_ids.csv'
else:
    DATA_ROOT       = os.path.abspath('../')
    RAW_TRAIN_PATH  = os.path.abspath('../data_raw/train.csv')
    EEG_DIR         = os.path.join(DATA_ROOT, 'train_eegs')
    SPEC_DIR        = os.path.join(DATA_ROOT, 'train_spectrograms')
    SAMPLE_IDS_PATH = os.path.abspath('../data_raw/workshop_sample_ids.csv')

VOTE_COLS   = ["seizure_vote", "lpd_vote", "gpd_vote", "lrda_vote", "grda_vote", "other_vote"]
LABEL_NAMES = ["Seizure", "LPD", "GPD", "LRDA", "GRDA", "Other"]
COLORS      = ["#d62728", "#ff7f0e", "#bcbd22", "#2ca02c", "#17becf", "#7f7f7f"]

print(f"Competition data path exists: {os.path.exists(DATA_ROOT)}")

## 1 · Dataset scale (full HMS competition data)

These numbers describe the full training set that participants join the competition for — this is *not* the workshop's 600-row subset.

In [ ]:
meta = pd.read_csv(RAW_TRAIN_PATH)

summary_rows = [
    ("Total labeled windows",         len(meta)),
    ("Unique patients",               meta["patient_id"].nunique()),
    ("Unique EEG recordings",         meta["eeg_id"].nunique()),
    ("Unique spectrogram recordings", meta["spectrogram_id"].nunique()),
    ("Avg labeled windows / patient",  round(len(meta) / meta["patient_id"].nunique(), 1)),
]
summary = pd.DataFrame(summary_rows, columns=["Metric", "Value"])
summary

## 2 · Expert consensus quality — not all labels are equally trustworthy

Each labeled window was reviewed by up to ~20 experts, who each cast one vote across the 6 categories. A window where the top category gets, say, 18/20 votes is a **confident** label. A window where the top category only gets 3/20 votes is a much **more ambiguous** one — the "ground truth" itself is uncertain.

In [ ]:
HIGH_THRESH = 10

max_vote   = meta[VOTE_COLS].max(axis=1)
total_vote = meta[VOTE_COLS].sum(axis=1)

high_mask = max_vote >= HIGH_THRESH   # matches the n_votes>=10 filter used in the paper pipeline
low_mask  = max_vote <  HIGH_THRESH

n_total, n_high, n_low = len(meta), high_mask.sum(), low_mask.sum()
print(f"Total labeled windows          : {n_total:>8,}  (100.0%)")
print(f"High consensus (max >= {HIGH_THRESH} votes): {n_high:>8,}  ({n_high/n_total:.1%})")
print(f"Low consensus  (max <  {HIGH_THRESH} votes): {n_low:>8,}  ({n_low/n_total:.1%})")

fig, ax = plt.subplots(figsize=(9, 4))
bins = range(0, int(max_vote.max()) + 2)
ax.hist(max_vote, bins=bins, color="steelblue", edgecolor="white", alpha=0.85)
ax.axvline(HIGH_THRESH, color="orange", lw=2, ls="--", label=f"Threshold = {HIGH_THRESH} (inclusive)")
ax.set_xlabel("Max single-category vote count (per window)")
ax.set_ylabel("Number of windows")
ax.set_title("Expert agreement distribution — many windows are genuinely ambiguous")
ax.legend()
plt.tight_layout()
plt.show()

## 3 · Class balance — the full dataset is heavily imbalanced

This is the imbalance that motivates 2-step training on the full dataset. The workshop subset (next section) is deliberately built to be balanced instead, so we can run fast, clean comparisons in the time we have.

In [ ]:
consensus_counts = meta["expert_consensus"].value_counts().reindex(LABEL_NAMES)

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(LABEL_NAMES, consensus_counts.values, color=COLORS, edgecolor="white")
ax.bar_label(bars, padding=3, fontsize=10, fontweight="bold")
ax.set_ylabel("Number of labeled windows")
ax.set_title(f"Full dataset class distribution — "
             f"{consensus_counts.max()/consensus_counts.min():.1f}x imbalance "
             f"({consensus_counts.idxmax()} vs {consensus_counts.idxmin()})")
plt.tight_layout()
plt.show()

print(consensus_counts)

In [ ]:
# ============ Is the high-quality (>=10 vote) subset also imbalanced? ============
# Reuses `high_mask` from the consensus-quality section above — no need to recompute.
high_quality_counts = meta.loc[high_mask, "expert_consensus"].value_counts().reindex(LABEL_NAMES)

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(LABEL_NAMES, high_quality_counts.values, color=COLORS, edgecolor="white")
ax.bar_label(bars, padding=3, fontsize=10, fontweight="bold")
ax.set_ylabel("Number of labeled windows")
ax.set_title(f"High-quality subset (max_vote >= {HIGH_THRESH}) class distribution — "
             f"{high_quality_counts.max()/high_quality_counts.min():.1f}x imbalance "
             f"({high_quality_counts.idxmax()} vs {high_quality_counts.idxmin()})")
plt.tight_layout()
plt.show()

print(high_quality_counts)
print(f"\n(for comparison, full dataset imbalance was "
      f"{consensus_counts.max()/consensus_counts.min():.1f}x)")

## 4 · Workshop subset vs full dataset

The workshop uses `workshop_sample_ids.csv` — a stratified, patient-level sample: 100 rows/class for training, 20 rows/class for validation, with zero patient overlap between the two splits.

In [ ]:
sample_ids = pd.read_csv(SAMPLE_IDS_PATH)
workshop_counts = sample_ids["expert_consensus"].value_counts().reindex(LABEL_NAMES)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

bars0 = axes[0].bar(LABEL_NAMES, consensus_counts.values, color=COLORS, edgecolor="white")
axes[0].bar_label(bars0, padding=3, fontsize=9, fontweight="bold")
axes[0].set_title(f"Full dataset (n={len(meta):,})\nheavily imbalanced")
axes[0].set_ylabel("Number of windows")

axes[1].bar(LABEL_NAMES, workshop_counts.values, color=COLORS, edgecolor="white")
axes[1].bar_label(axes[1].containers[0], padding=3, fontsize=10, fontweight="bold")
axes[1].set_title(f"Workshop subset (n={len(sample_ids):,})\nbalanced by design")

plt.tight_layout()
plt.show()

print(f"Unique patients in workshop subset: {sample_ids['patient_id'].nunique()} "
      f"(out of {meta['patient_id'].nunique():,} total)")
train_p = set(sample_ids[sample_ids['split']=='train']['patient_id'])
val_p   = set(sample_ids[sample_ids['split']=='val']['patient_id'])
print(f"Train/val patient overlap: {len(train_p & val_p)} (should be 0)")

## 5 · Why we split by `patient_id`, not by row

Some patients contribute many labeled windows. If we split randomly by row, the same patient's EEG could appear in both train and validation — the model could then "recognize" that patient's individual signal characteristics rather than learning to generalize. Splitting by patient closes this leak.

In [ ]:
windows_per_patient = meta.groupby("patient_id").size()

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(windows_per_patient, bins=50, color="steelblue", edgecolor="white")
ax.set_xlabel("Labeled windows per patient")
ax.set_ylabel("Number of patients")
ax.set_title(f"Some patients contribute far more windows than others "
             f"(median={windows_per_patient.median():.0f}, max={windows_per_patient.max()})")
plt.tight_layout()
plt.show()

## 6 · Where in the recording is the labeled window?

`eeg_label_offset_seconds` tells you where the 50-second labeled segment sits within the full EEG recording. It is **not** always near the center of the file — using a fixed center-crop instead of this offset would frequently extract the wrong segment.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(meta["eeg_label_offset_seconds"], bins=60, color="steelblue", edgecolor="white")
ax.set_xlabel("eeg_label_offset_seconds")
ax.set_ylabel("Number of labeled windows")
ax.set_title("Labeled window position within the recording — not concentrated at the center")
plt.tight_layout()
plt.show()

print(meta["eeg_label_offset_seconds"].describe())

## 7 · Raw signal amplitude — the numbers behind the mu-law calibration

Mu-law encoding assumes its input is roughly within `[-1, 1]` before compression. Here's what the actual raw signal amplitude looks like (after bipolar montage), sampled from the workshop training set — this is exactly the distribution used to calibrate `MULAW_SCALE` in the 1D CNN notebook.

In [ ]:
CHANNELS = ['Fp1','F3','C3','P3','F7','T3','T5','O1','Fz','Cz','Pz',
            'Fp2','F4','C4','P4','F8','T4','T6','O2']

def bipolar_montage(eeg_20ch: np.ndarray) -> np.ndarray:
    # eeg_20ch: (T, 19) in the CHANNELS order above (EKG excluded)
    idx = {c: i for i, c in enumerate(CHANNELS)}
    return np.stack([
        eeg_20ch[:, idx['Fp1']] - eeg_20ch[:, idx['T3']],
        eeg_20ch[:, idx['T3']]  - eeg_20ch[:, idx['O1']],
        eeg_20ch[:, idx['Fp1']] - eeg_20ch[:, idx['C3']],
        eeg_20ch[:, idx['C3']]  - eeg_20ch[:, idx['O1']],
        eeg_20ch[:, idx['Fp2']] - eeg_20ch[:, idx['C4']],
        eeg_20ch[:, idx['C4']]  - eeg_20ch[:, idx['O2']],
        eeg_20ch[:, idx['Fp2']] - eeg_20ch[:, idx['T4']],
        eeg_20ch[:, idx['T4']]  - eeg_20ch[:, idx['O2']],
    ], axis=1)  # (T, 8)

train_eeg_ids = sample_ids[sample_ids['split'] == 'train']['eeg_id'].unique()[:30]
abs_vals = []
for eeg_id in train_eeg_ids:
    path = os.path.join(EEG_DIR, f"{int(eeg_id)}.parquet")
    if not os.path.exists(path):
        continue
    eeg = pd.read_parquet(path, columns=CHANNELS).to_numpy(dtype=np.float32)
    eeg = np.nan_to_num(eeg, nan=0.0, posinf=0.0, neginf=0.0)
    bp  = bipolar_montage(eeg)
    abs_vals.append(np.abs(bp).ravel())
abs_vals = np.concatenate(abs_vals)

p50, p99 = np.percentile(abs_vals, [50, 99])
print(f"Bipolar signal amplitude — median: {p50:.1f}, 99th percentile: {p99:.1f}, max: {abs_vals.max():.1f}")
print(f"(units are the raw EEG amplitude units in the parquet files, roughly microvolts)")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(np.clip(abs_vals, 0, 500), bins=80, color="steelblue", edgecolor="white")
ax.axvline(p99, color="red", ls="--", label=f"99th percentile = {p99:.0f}")
ax.set_xlabel("|bipolar signal amplitude| (clipped at 500 for display)")
ax.set_ylabel("Count")
ax.set_title("This is the real scale mu-law needs calibrating against — not a fixed constant")
ax.legend()
plt.tight_layout()
plt.show()

## 8 · Electrode layout — what bipolar montage keeps, and what it drops

The standard double-banana bipolar montage used in the 1D CNN notebook only differences 8 of the 19 EEG electrodes. The midline electrodes (Fz, Cz, Pz) and several others are not used at all — this matters for classes like LRDA vs GRDA, whose main distinguishing feature is *spatial extent* (lateralized vs generalized).

In [ ]:
# Approximate 10-20 system positions (schematic, top-down view, nose at top)
POSITIONS = {
    'Fp1': (-0.3, 0.9),  'Fp2': (0.3, 0.9),
    'F7':  (-0.9, 0.5),  'F3':  (-0.5, 0.5), 'Fz': (0.0, 0.5), 'F4': (0.5, 0.5), 'F8': (0.9, 0.5),
    'T3':  (-1.0, 0.0),  'C3':  (-0.5, 0.0), 'Cz': (0.0, 0.0), 'C4': (0.5, 0.0), 'T4': (1.0, 0.0),
    'T5':  (-0.9, -0.5), 'P3':  (-0.5, -0.5), 'Pz': (0.0, -0.5), 'P4': (0.5, -0.5), 'T6': (0.9, -0.5),
    'O1':  (-0.3, -0.9), 'O2':  (0.3, -0.9),
}
MONTAGE_ELECTRODES = {'Fp1', 'T3', 'O1', 'C3', 'Fp2', 'C4', 'T4', 'O2'}

fig, ax = plt.subplots(figsize=(6.5, 6.5))
head = plt.Circle((0, 0), 1.05, fill=False, color='black', lw=1.5)
ax.add_patch(head)
ax.plot([0], [1.15], marker='^', color='black', markersize=10)  # nose marker

for ch, (x, y) in POSITIONS.items():
    used = ch in MONTAGE_ELECTRODES
    ax.scatter(x, y, s=500, color=('#d62728' if used else '#bbbbbb'),
               edgecolor='black', zorder=3)
    ax.text(x, y, ch, ha='center', va='center', fontsize=8, fontweight='bold',
            color='white' if used else 'black', zorder=4)

ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3)
ax.set_aspect('equal'); ax.axis('off')
ax.set_title("Red = used by bipolar montage (8 electrodes)\nGray = not used (11 electrodes, incl. Fz/Cz/Pz midline)")
plt.tight_layout()
plt.show()

## 9 · One EEG + spectrogram example per class

A visual sanity check — one representative labeled window per consensus class, from the workshop subset.

In [ ]:
# workshop_sample_ids.csv only carries IDs + label + split — merge back to the
# full train.csv (already loaded as `meta`) to recover eeg_label_offset_seconds,
# spectrogram_label_offset_seconds, and the vote columns.
sample_ids_full = sample_ids.merge(
    meta[["eeg_id", "eeg_sub_id", "eeg_label_offset_seconds",
          "spectrogram_label_offset_seconds"] + VOTE_COLS],
    on=["eeg_id", "eeg_sub_id"],
    how="left",
)
sample_ids_full["max_vote"] = sample_ids_full[VOTE_COLS].max(axis=1)

# Pick the HIGHEST-confidence (highest max_vote) example per class, not just the first
# row encountered — section 2 above showed many labels are ambiguous, so an arbitrary
# pick could easily land on a low-consensus window and undercut this section's own point.
sample_table = (
    sample_ids_full[sample_ids_full['split'] == 'train']
    .sort_values('max_vote', ascending=False)
    .groupby('expert_consensus')
    .first()
    .reindex(LABEL_NAMES)
    .reset_index()
)
sample_table


In [ ]:
FS = 200
WINDOW_SEC = 50

fig, axes = plt.subplots(len(sample_table), 1, figsize=(16, 3.2 * len(sample_table)), sharex=False)

for ax, (_, row) in zip(axes, sample_table.iterrows()):
    eeg_id = int(row['eeg_id'])
    offset = float(row['eeg_label_offset_seconds'])
    eeg = pd.read_parquet(os.path.join(EEG_DIR, f"{eeg_id}.parquet"), columns=CHANNELS).to_numpy(dtype=np.float32)
    eeg = np.nan_to_num(eeg, nan=0.0)
    time = np.arange(len(eeg)) / FS

    SCALE = 150
    for i, ch in enumerate(CHANNELS):
        y_off = (len(CHANNELS) - 1 - i) * SCALE
        sig = np.clip(eeg[:, i], -500, 500)
        ax.plot(time, sig + y_off, lw=0.3, color='steelblue')

    ax.axvspan(offset, offset + WINDOW_SEC, color='orange', alpha=0.15, label='Labeled window')
    ax.set_xlim(max(0, offset - 30), min(time[-1], offset + WINDOW_SEC + 30))
    ax.set_yticks([])
    ax.set_title(f"{row['expert_consensus']}  —  eeg_id={eeg_id}, offset={offset:.0f}s")

axes[0].legend(loc='upper right', fontsize=8)  # one legend is enough — same label on every subplot

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(sample_table), figsize=(4 * len(sample_table), 4))

SPEC_WIDTH = 300  # 300 columns = 600 seconds at 2s/column — matches the 2D CNN notebook's SpectrogramDataset

for ax, (_, row) in zip(axes, sample_table.iterrows()):
    spec_id = int(row['spectrogram_id'])
    offset  = float(row['spectrogram_label_offset_seconds'])

    spec_df = pd.read_parquet(os.path.join(SPEC_DIR, f"{spec_id}.parquet"))
    spec_df = spec_df.drop(columns=['time'], errors='ignore').fillna(0)
    arr = spec_df.to_numpy(dtype=np.float32).T   # (400, total_time) -> 4 chains x 100 freq bins

    # crop to the labeled 600-second window, same as SpectrogramDataset in the 2D CNN notebook —
    # otherwise this shows the *entire* recording, which can span multiple labeled windows
    col_start = int(offset // 2)
    window = arr[:, col_start:col_start + SPEC_WIDTH]
    if window.shape[1] < SPEC_WIDTH:
        window = np.pad(window, ((0, 0), (0, SPEC_WIDTH - window.shape[1])), mode='constant')

    window = np.log1p(np.clip(window, 0, None))
    ax.imshow(window, aspect='auto', origin='lower', cmap='viridis')
    ax.set_title(f"{row['expert_consensus']}\noffset={offset:.0f}s", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()


## Wrap-up — how this connects to the rest of the day, we are not addressing every point with todays hands-on

- **Consensus quality (§2)** is why the paper's benchmark filters to `n_votes >= 10` before
  training, and why 2-step training exists at all — Step 1 sees the full noisy data, Step 2
  corrects on the clean subset.
- **Patient-level split (§5)** is why every workshop/paper notebook splits by `patient_id`,
  never by row — otherwise validation numbers would be inflated by leakage.
- **Window position (§6)** is why every notebook uses `eeg_label_offset_seconds` /
  `spectrogram_label_offset_seconds` to crop, instead of a fixed center-crop.
- **Signal amplitude (§7)** is exactly what calibrates `MULAW_SCALE` in the 1D CNN notebook —
  a fixed constant would saturate on the real amplitude range you just saw.
- **Electrode layout (§8)** is why bipolar montage alone struggles with LRDA vs GRDA — both
  are largely defined by *spatial extent* (lateralized vs generalized), and the dropped
  midline electrodes (Fz/Cz/Pz) carry some of that information.

Next up: XGBoost (hand-crafted features, no montage) → 1D CNN (naive vs domain-informed
preprocessing) → 2D CNN (pretrained EfficientNet on spectrograms).
